# 03 · Comparación de modelos no lineales sin fuga de información

Este notebook compara modelos de árboles para predecir `risk_class_1h`: 0 = estable, 1 = riesgo de vaciado y 2 = riesgo de saturación. Mantiene exactamente las mismas particiones temporales y el mismo protocolo sin fuga del notebook 02.

Los modelos comparados son Random Forest, Extra Trees y LightGBM cuando este último esté instalado.

## Dependencias

Necesitas `scikit-learn`. LightGBM es opcional: si no está disponible, el notebook continúa con los dos modelos de árboles de scikit-learn.

In [ ]:

# %pip install scikit-learn
%pip install lightgbm

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# LightGBM se importa de forma opcional para que el notebook funcione aunque no esté instalado.
try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print('LightGBM no está instalado: se compararán Random Forest y Extra Trees.')

# Localizamos las carpetas del proyecto.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'Datos modelado').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURE_DIR = PROJECT_ROOT / 'Datos modelado' / 'estacion_hora_features'
RESULTS_DIR = PROJECT_ROOT / 'Datos modelado' / 'resultados_modelos'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = 'risk_class_1h'
RANDOM_STATE = 42

# La muestra hace viable una primera comparación. Usa None para entrenar con todo train.
MAX_TRAIN_ROWS = 500_000

# El test se bloquea hasta seleccionar el modelo mediante validation.
EVALUATE_FINAL_TEST = True


## Variables y carga eficiente

La lista excluye cualquier variable futura o que defina directamente la etiqueta. La carga se hace por bloques y el muestreo se aplica solo a train; validation y test se preservan completos.

In [ ]:
NUMERIC_FEATURES = [
    'capacity', 'bikes_available', 'docks_available', 'reservations_count',
    'occupancy_ratio', 'light', 'weather_available',
    'uv_radiation_median_mw_m2', 'wind_speed_median_m_s',
    'wind_direction_sin_mean', 'wind_direction_cos_mean',
    'temperature_median_c', 'relative_humidity_median_pct',
    'barometric_pressure_median_mb', 'solar_radiation_median_w_m2',
    'precipitation_mean_l_m2', 'precipitation_max_l_m2',
    'n_temperature', 'n_relative_humidity', 'n_precipitation',
    'hour', 'day_of_week', 'month', 'week_of_year',
    'occupancy_ratio_lag_1h', 'occupancy_ratio_lag_2h', 'occupancy_ratio_lag_24h',
    'bikes_available_lag_1h', 'bikes_available_lag_2h', 'bikes_available_lag_24h',
    'net_flow_lag_1h', 'net_flow_lag_2h', 'net_flow_lag_24h',
    'departures_count_lag_1h', 'departures_count_lag_2h', 'departures_count_lag_24h',
    'arrivals_count_lag_1h', 'arrivals_count_lag_2h', 'arrivals_count_lag_24h',
    'occupancy_ratio_mean_previous_3h', 'occupancy_ratio_mean_previous_24h',
    'net_flow_mean_previous_3h', 'net_flow_mean_previous_24h',
    'departures_count_mean_previous_3h', 'departures_count_mean_previous_24h',
    'arrivals_count_mean_previous_3h', 'arrivals_count_mean_previous_24h',
]
CATEGORICAL_FEATURES = ['station_id', 'tipo_dia']
COLUMNS_TO_LOAD = NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET, 'dataset_split']
feature_files = sorted(FEATURE_DIR.glob('estacion_hora_features_*.csv'))
assert len(feature_files) == 48, f'Se esperaban 48 particiones y hay {len(feature_files)}'


def count_train_rows() -> int:
    """Cuenta filas elegibles de train sin cargar todas las variables."""
    total = 0
    for file_path in feature_files:
        for chunk in pd.read_csv(file_path, usecols=[TARGET, 'dataset_split'], chunksize=300_000):
            total += int((chunk['dataset_split'].eq('train') & chunk[TARGET].notna()).sum())
    return total


def load_splits(train_fraction: float) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Lee por bloques y devuelve train muestreado, validation completo y test completo."""
    train_parts, validation_parts, test_parts = [], [], []
    for file_path in feature_files:
        for chunk in pd.read_csv(file_path, usecols=COLUMNS_TO_LOAD, chunksize=200_000, low_memory=False):
            is_eligible = chunk['dataset_split'].isin(['train', 'validation', 'test']) & chunk[TARGET].notna()
            chunk = chunk.loc[is_eligible].copy()
            if chunk.empty:
                continue
            train_chunk = chunk.loc[chunk['dataset_split'].eq('train')]
            if not train_chunk.empty:
                # Se muestrea dentro de cada clase para preservar aproximadamente su distribución.
                if train_fraction < 1:
                    train_chunk = train_chunk.groupby(TARGET, group_keys=False).sample(frac=train_fraction, random_state=RANDOM_STATE)
                train_parts.append(train_chunk)
            validation_parts.append(chunk.loc[chunk['dataset_split'].eq('validation')])
            test_parts.append(chunk.loc[chunk['dataset_split'].eq('test')])
    return (
        pd.concat(train_parts, ignore_index=True),
        pd.concat(validation_parts, ignore_index=True),
        pd.concat(test_parts, ignore_index=True),
    )


train_total = count_train_rows()
train_fraction = 1.0 if MAX_TRAIN_ROWS is None else min(1.0, MAX_TRAIN_ROWS / train_total)
train, validation, test = load_splits(train_fraction)

for frame in [train, validation, test]:
    frame[TARGET] = frame[TARGET].astype('int8')

print({'train': len(train), 'validation': len(validation), 'test': len(test)})
print('Distribución de clases en train:')
print(train[TARGET].value_counts(normalize=True).sort_index())

## Preprocesamiento ajustado solo con train

La siguiente celda es el control central contra fuga de información: `fit_transform` se aplica solo a `X_train`. Validation y test pasan únicamente por `transform`, por lo que no influyen en medianas ni categorías aprendidas.

In [ ]:
X_train = train[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_train = train[TARGET]
X_validation = validation[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_validation = validation[TARGET]
X_test = test[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_test = test[TARGET]

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
])
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('one_hot', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(transformers=[
    ('numeric', numeric_pipeline, NUMERIC_FEATURES),
    ('categorical', categorical_pipeline, CATEGORICAL_FEATURES),
])

# El ajuste ocurre solo con train. Este objeto no se vuelve a ajustar con validation ni con test.
X_train_ready = preprocessor.fit_transform(X_train)
X_validation_ready = preprocessor.transform(X_validation)
X_test_ready = preprocessor.transform(X_test)

print('Dimensión tras transformar:', X_train_ready.shape)

In [ ]:
def evaluate(model, features, target, name: str) -> dict:
    """Evalúa cada modelo con métricas robustas ante clases desbalanceadas."""
    prediction = model.predict(features)
    print(f'\n--- {name} ---')
    print('Balanced accuracy:', round(balanced_accuracy_score(target, prediction), 4))
    print('F1 macro:', round(f1_score(target, prediction, average='macro'), 4))
    print('Matriz de confusión (filas: real; columnas: predicción):')
    print(confusion_matrix(target, prediction, labels=[0, 1, 2]))
    print(classification_report(target, prediction, labels=[0, 1, 2], target_names=['estable', 'vaciado', 'saturación'], zero_division=0))
    return {
        'model': name,
        'balanced_accuracy': balanced_accuracy_score(target, prediction),
        'f1_macro': f1_score(target, prediction, average='macro'),
    }


models = {
    # Random Forest combina árboles entrenados con muestras y variables aleatorias.
    'Random Forest': RandomForestClassifier(
        n_estimators=250, max_features='sqrt', min_samples_leaf=5,
        class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE,
    ),
    # Extra Trees introduce aún más aleatoriedad en las divisiones y suele ser rápido para explorar.
    'Extra Trees': ExtraTreesClassifier(
        n_estimators=250, max_features='sqrt', min_samples_leaf=5,
        class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE,
    ),
}

if LIGHTGBM_AVAILABLE:
    # LightGBM aplica boosting: cada árbol intenta corregir errores de los anteriores.
    models['LightGBM'] = LGBMClassifier(
        objective='multiclass', num_class=3, n_estimators=300, learning_rate=0.05,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1,
    )

fitted_models = {}
validation_results = []
for name, model in models.items():
    print(f'Entrenando: {name}')
    model.fit(X_train_ready, y_train)
    fitted_models[name] = model
    validation_results.append(evaluate(model, X_validation_ready, y_validation, name))

results_validation = pd.DataFrame(validation_results).sort_values('f1_macro', ascending=False)
results_validation.to_csv(RESULTS_DIR / 'comparacion_no_lineales_validation.csv', index=False, encoding='utf-8-sig')
results_validation

In [ ]:
# Analizamos las variables más influyentes del mejor modelo de validation.
best_name = results_validation.iloc[0]['model']
best_model = fitted_models[best_name]

if hasattr(best_model, 'feature_importances_'):
    feature_names = preprocessor.get_feature_names_out()
    importance = pd.DataFrame({
        'feature': feature_names,
        'importance': best_model.feature_importances_,
    }).sort_values('importance', ascending=False)
    importance.head(20)
else:
    print(f'{best_name} no expone importancias de variables en este formato.')

In [ ]:
# El test final se usa una sola vez, después de seleccionar el mejor modelo mediante validation.
if EVALUATE_FINAL_TEST:
    final_results = evaluate(best_model, X_test_ready, y_test, f'Test final: {best_name}')
    pd.DataFrame([final_results]).to_csv(RESULTS_DIR / 'mejor_modelo_test_final.csv', index=False, encoding='utf-8-sig')
    pd.DataFrame([final_results])
else:
    print('Test bloqueado. Cambia EVALUATE_FINAL_TEST a True solo cuando elijas el modelo con validation.')

## Próximo paso

Después de elegir un modelo por validation y medirlo una vez en test, el siguiente notebook puede centrarse en interpretabilidad: importancia de variables, análisis de errores por hora y estación, y preparación de recomendaciones operativas.